In [2]:
import os
from pyspark.sql import SparkSession

#os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
#os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

spark = SparkSession.builder \
    .appName("GarbageStats") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Raw DATA

In [3]:
df_train = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/train")

df_test = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/test")

In [4]:
from pyspark.sql.functions import lit, element_at, size, count, when, col, min, max, avg, split

In [5]:
df_train = df_train.withColumn("split", lit("train"))
df_test = df_test.withColumn("split", lit("test"))

df_raw = df_train.unionByName(df_test)

df_raw = df_raw.withColumn(
    "category",
    element_at(split("path", "/"), -2)
)

In [6]:
df_raw.count()

10464

In [7]:
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

+----+----------------+------+-------+-----+--------+
|path|modificationTime|length|content|split|category|
+----+----------------+------+-------+-----+--------+
|   0|               0|     0|      0|    0|       0|
+----+----------------+------+-------+-----+--------+



In [8]:
df_raw.groupBy("split").count().show()

+-----+-----+
|split|count|
+-----+-----+
|train| 7324|
| test| 3140|
+-----+-----+



In [9]:
df_raw.groupBy("category").count().orderBy("count", ascending=False).show()

+-------------+-----+
|     category|count|
+-------------+-----+
|        glass| 2649|
|biodegradable| 2220|
|        paper| 1746|
|    cardboard| 1439|
|        metal| 1248|
|      plastic| 1162|
+-------------+-----+



In [10]:
df_raw.select("length").describe().show()

+-------+------------------+
|summary|            length|
+-------+------------------+
|  count|             10464|
|   mean|19151.321769877675|
| stddev| 9474.638532461628|
|    min|              4188|
|    max|             67006|
+-------+------------------+



In [11]:
df_raw.groupBy("path").count().filter("count > 1").show() #doublons

+----+-----+
|path|count|
+----+-----+
+----+-----+



# DATA Transformées

In [12]:
df_train = spark.read.parquet("../data/train_data.parquet")
df_test = spark.read.parquet("../data/test_data.parquet")

In [13]:
train_count = df_train.count()
test_count = df_test.count()

total = train_count + test_count

print("Train :", train_count)
print("Test :", test_count)
print("Total :", total)

Train : 7324
Test : 3140
Total : 10464


In [14]:
df_train.printSchema()
df_test.printSchema()

root
 |-- x_train: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_train: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)

root
 |-- x_test: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_test: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)



In [15]:
df_train.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_train.columns
]).show()

+-------+-------+-----+
|x_train|y_train|class|
+-------+-------+-----+
|      0|      0|    0|
+-------+-------+-----+



In [16]:
df_test.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_test.columns
]).show()

+------+------+-----+
|x_test|y_test|class|
+------+------+-----+
|     0|     0|    0|
+------+------+-----+



In [17]:
df_train.groupBy("class").count().orderBy("count", ascending=False).show()
df_test.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass| 1854|
|biodegradable| 1554|
|        paper| 1223|
|    cardboard| 1006|
|        metal|  874|
|      plastic|  813|
+-------------+-----+

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass|  795|
|biodegradable|  666|
|        paper|  523|
|    cardboard|  433|
|        metal|  374|
|      plastic|  349|
+-------------+-----+



In [18]:
df_train = df_train.withColumn("height", size("x_train"))
df_train = df_train.withColumn("width", size(col("x_train")[0]))

df_train.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 7324|
+------+-----+-----+



In [19]:
df_test = df_test.withColumn("height", size("x_test"))
df_test = df_test.withColumn("width", size(col("x_test")[0]))

df_test.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 3140|
+------+-----+-----+



In [20]:
df_train.withColumn("label_size", size("y_train")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 7324|
+----------+-----+



In [21]:
df_test.withColumn("label_size", size("y_test")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 3140|
+----------+-----+



# Prédiction DATA

In [22]:
df_pred = spark.read.parquet("../data/prediction_data.parquet")

In [23]:
df_pred.count()

6

In [24]:
df_pred.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|      plastic|    1|
|        metal|    1|
|biodegradable|    1|
|        paper|    1|
|    cardboard|    1|
|        glass|    1|
+-------------+-----+



In [25]:
df_pred.groupBy("file").count().filter("count > 1").show()

+----+-----+
|file|count|
+----+-----+
+----+-----+



In [26]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).show()

+----+----------+--------+-----+----------+
|file|prediction|class_id|class|confidence|
+----+----------+--------+-----+----------+
|   0|         0|       0|    0|         0|
+----+----------+--------+-----+----------+



In [27]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .show()

+-------------+-----+------------------+
|        class|count|        percentage|
+-------------+-----+------------------+
|      plastic|    1|16.666666666666664|
|        metal|    1|16.666666666666664|
|biodegradable|    1|16.666666666666664|
|        paper|    1|16.666666666666664|
|    cardboard|    1|16.666666666666664|
|        glass|    1|16.666666666666664|
+-------------+-----+------------------+



In [28]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .show()

+-------------+-------------------+
|        class|     avg_confidence|
+-------------+-------------------+
|biodegradable|  0.416423499584198|
|        glass| 0.4031400680541992|
|      plastic| 0.3986396789550781|
|        paper| 0.3605785667896271|
|        metal|0.36037102341651917|
|    cardboard| 0.2772018015384674|
+-------------+-------------------+



In [29]:
df_pred.filter(col("confidence") < 0.5).show()

+--------------------+--------------------+--------+-------------+-------------------+
|                file|          prediction|class_id|        class|         confidence|
+--------------------+--------------------+--------+-------------+-------------------+
|GettyImages-48762...|[0.02559871412813...|       6|      plastic| 0.3986396789550781|
|Encart haut droit...|[0.07713691890239...|       4|        metal|0.36037102341651917|
|1687380-bouteille...|[0.41642349958419...|       1|biodegradable|  0.416423499584198|
|dechets-alimentai...|[0.33676996827125...|       5|        paper| 0.3605785667896271|
| recyclage-verre.jpg|[0.08144132047891...|       2|    cardboard| 0.2772018015384674|
|       711339_d1.jpg|[0.00969277881085...|       3|        glass| 0.4031400680541992|
+--------------------+--------------------+--------+-------------+-------------------+



In [30]:
df_pred.select("confidence").describe().show()

+-------+-------------------+
|summary|         confidence|
+-------+-------------------+
|  count|                  6|
|   mean|0.36939243972301483|
| stddev|0.05072358049167791|
|    min| 0.2772018015384674|
|    max|  0.416423499584198|
+-------+-------------------+



In [31]:
df_pred.orderBy("confidence", ascending=False).show(10)

+--------------------+--------------------+--------+-------------+-------------------+
|                file|          prediction|class_id|        class|         confidence|
+--------------------+--------------------+--------+-------------+-------------------+
|1687380-bouteille...|[0.41642349958419...|       1|biodegradable|  0.416423499584198|
|       711339_d1.jpg|[0.00969277881085...|       3|        glass| 0.4031400680541992|
|GettyImages-48762...|[0.02559871412813...|       6|      plastic| 0.3986396789550781|
|dechets-alimentai...|[0.33676996827125...|       5|        paper| 0.3605785667896271|
|Encart haut droit...|[0.07713691890239...|       4|        metal|0.36037102341651917|
| recyclage-verre.jpg|[0.08144132047891...|       2|    cardboard| 0.2772018015384674|
+--------------------+--------------------+--------+-------------+-------------------+



# Ecriture parquet 

In [32]:
df_pred.select(count("*").alias("total")) \
    .write.mode("overwrite") \
    .parquet("../data/stats/total.parquet")

In [33]:
df_pred.groupBy("class") \
    .count() \
    .orderBy("count", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/count_by_class.parquet")

In [34]:
df_pred.groupBy("file") \
    .count() \
    .filter("count > 1") \
    .write.mode("overwrite") \
    .parquet("../data/stats/duplicates.parquet")

In [35]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).write.mode("overwrite") \
 .parquet("../data/stats/nulls.parquet")

In [36]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/percentage_by_class.parquet")

In [37]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_by_class.parquet")

In [38]:
df_pred.filter(col("confidence") < 0.5) \
    .write.mode("overwrite") \
    .parquet("../data/stats/low_confidence.parquet")

In [39]:
df_pred.select("confidence") \
    .describe() \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_stats.parquet")

In [40]:
df_pred.orderBy("confidence", ascending=False) \
    .limit(10) \
    .write.mode("overwrite") \
    .parquet("../data/stats/top_predictions.parquet")